In [1]:
import pandas as pd

selected_df_notes= pd.read_csv("data/selected_labeled_notes.csv")
print(selected_df_notes.shape)

(4989, 9)


exploring selected_notes files

In [2]:
code_count = selected_df_notes["matched_code"].value_counts()
print(code_count)

matched_code
T502X2S    177
V123XXS    130
O368325    125
O368323    108
T502X6S     71
          ... 
O368314      1
I70223       1
S066X8A      1
P0504        1
P0506        1
Name: count, Length: 847, dtype: int64


In [3]:
for min_count in [5, 10, 20]:
    filtered = code_count[code_count >= min_count]
    print(f"min_count={min_count}: {filtered.shape[0]} classes, covers {filtered.sum()} notes")

min_count=5: 232 classes, covers 3569 notes
min_count=10: 102 classes, covers 2701 notes
min_count=20: 50 classes, covers 2020 notes


In [4]:
# making training data 
frequent_codes = code_count[code_count>= 5].index

final_training_data= selected_df_notes[selected_df_notes["matched_code"].isin(frequent_codes)].copy()
print(final_training_data.shape)

(3569, 9)


In [5]:
# labeling unique code with a index or label  and sorting list base on unique codes
unique_codes= sorted(final_training_data["matched_code"].unique())
code_to_label={}
for i , code in enumerate(unique_codes):
    code_to_label[code]=i
final_training_data["label"]= final_training_data["matched_code"].map(code_to_label)



In [6]:
print(final_training_data[["matched_code", "label"]].head())
print(final_training_data["label"].nunique())
print(final_training_data["label"].max())

  matched_code  label
0      O361132     93
1      T502X2S    163
2        Z6855    229
6         Z421    226
9       O30831     78
232
231


 # train test split


In [7]:
from sklearn.model_selection import train_test_split

train_data, val_data =train_test_split(final_training_data, test_size=0.2,stratify= final_training_data["label"],random_state=42)
print(train_data.shape)
print(val_data.shape)


from transformers import AutoTokenizer
tokenizer= AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.1")

train_token = tokenizer(list(train_data["combined_text"]), padding="max_length", truncation=True, max_length=256)
val_token = tokenizer(list(val_data["combined_text"]), padding="max_length", truncation=True, max_length=256)

print(train_token.keys())
print(len(train_token["input_ids"]))
print(len(val_token["input_ids"]))

(2855, 10)
(714, 10)


/opt/miniconda3/envs/dl_projects/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])
2855
714


pytorch dataset

In [8]:
import torch
from torch.utils.data import Dataset
class ICD_Dataset(Dataset):
    def __init__(self, encoding,labels):
        self.encoding = encoding
        self.labels = labels
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self,idx):
        
        item = {key: torch.tensor(value[idx]) for key,value in self.encoding.items()}
        
        item["labels"] = torch.tensor(self.labels[idx])
        return item

In [9]:
train_dataset= ICD_Dataset(train_token, list(train_data["label"]))
val_dataset= ICD_Dataset(val_token, list(val_data["label"]))
print(len(train_dataset))
print(train_dataset[0])

2855
{'input_ids': tensor([  101, 10211,  3113, 27231,  1111,   170,  5899,   118,  1214,   118,
         1385,   170,  2087, 15353,  1179,   118,  1821, 26237,  1389,  2130,
        11534,  1114,  2076,   123, 17972,  1107,  2278,   119, 23481,   131,
          117,  1103,  5351,  1110,   170,  5899,   118,  1214,   118,  1385,
          170,  2087, 15353,  1179,   118,  1821, 26237,  1389,  2130,  1114,
          170,  2191,  2103,  3976,  1104,   126,  2555,   124,  4519,  1105,
         2841,  1104, 21764,  6549,   119,  1131,  1108, 11534,  1114,  2076,
          123, 17972,  1107,  2278,   119,  1131,  1110,  1136,  1155, 26949,
         1106,  1251, 26016,   119,   117, 17972, 23897,   131,   117,  1123,
        17972, 23897,  1511, 21820, 19001, 26825,  3102,   120,  1476,   117,
         3140,  2338,  1120,  6462,  1105,  1659,  2338,  1120, 20644,   119,
         1145,  1899, 13199,  1394,  2260, 17713,  1120, 20644,   119,   117,
         1168, 23897,   131,   117,  1168, 23

# bioBERT model

In [10]:
import torch
from transformers import AutoModelForSequenceClassification

device= torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.1",
    num_labels=232
)
model.to(device)
print(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


mps


dataloader

In [11]:

from torch.utils.data import DataLoader

train_loader= DataLoader(train_dataset,batch_size=16, shuffle=True)
val_loader= DataLoader(val_dataset , batch_size=16, shuffle=True)

print(len(train_loader))
print(len(val_loader))

179
45


In [12]:
from torch.optim import AdamW

optimizer= AdamW(model.parameters(), lr = 2e-5)

In [13]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_data["label"]),
    y=train_data["label"]
)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(class_weights_tensor.shape)
print(class_weights_tensor[:5])


torch.Size([232])
tensor([3.0765, 2.4612, 0.6477, 0.4395, 0.5594], device='mps:0')


In [ ]:
import torch.nn as nn

loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

model.train()
epochs = 2
for epoch in range(epochs):
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )   # labels yahan NAHI diye — loss khud calculate karenge
        logits = outputs.logits
        loss = loss_fn(logits, labels)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    
    print(f"Epoch {epoch+1} (weighted loss, total_epochs_trained={total_epochs_trained}): avg loss = {total_loss / len(train_loader)}")
    

Epoch 1 (weighted loss, total_epochs_trained=51): avg loss = 0.8264552371461964
Epoch 2 (weighted loss, total_epochs_trained=56): avg loss = 0.6853487205238982


total epochs trained- 13

In [25]:
model.eval()
correct=0
total=0

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels = batch["labels"].to(device)
        
        output=model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        logits = output.logits
        prediction= torch.argmax(logits,dim=1)
        correct += (prediction == labels).sum().item()
        total += labels.size(0)
        
accuracy = correct / total
print(f"Validation accuracy: {accuracy:.4f}")

Validation accuracy: 0.8655


In [26]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

train_accuracy = correct / total
print(f"Training accuracy: {train_accuracy:.4f}")

Training accuracy: 0.9825


# experiment with freezing layer

In [19]:
import torch
from transformers import AutoModelForSequenceClassification

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Fresh model — purane 14-epoch wale model se bilkul alag copy
model_frozen = AutoModelForSequenceClassification.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.1",
    num_labels=232,
    attn_implementation="eager"    # <-- naya add kiya
)
model_frozen.to(device)

for param in model_frozen.bert.embeddings.parameters():
    param.requires_grad = False

for layer in model_frozen.bert.encoder.layer[:6]:
    for param in layer.parameters():
        param.requires_grad = False

trainable = sum(p.numel() for p in model_frozen.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_frozen.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable params: 43,296,232 / 108,488,680


In [ ]:
from torch.optim import AdamW

# sirf trainable parameters optimizer ko do — frozen wale ko chhedne ki zaroorat nahi
optimizer_frozen = AdamW(
    filter(lambda p: p.requires_grad, model_frozen.parameters()),
    lr=2e-5
)

model_frozen.train()
epochs = 3
for epoch in range(epochs):
    total_loss = 0
    for batch in train_loader:
        optimizer_frozen.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels = batch["labels"].to(device)

        outputs = model_frozen(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()
        optimizer_frozen.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}: avg loss = {total_loss / len(train_loader)}")
    
    
    

In [ ]:
model_frozen.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels = batch["labels"].to(device)

        outputs = model_frozen(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"Frozen model validation accuracy (after 3 epochs): {accuracy:.4f}")

# classification report of without freeze model

In [28]:
from sklearn.metrics import classification_report

model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

report = classification_report(all_labels, all_predictions, zero_division=0)
print(report)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1
           2       0.83      1.00      0.91         5
           3       0.70      1.00      0.82         7
           4       1.00      0.50      0.67         6
           5       1.00      0.50      0.67         2
           6       0.50      1.00      0.67         2
           7       1.00      1.00      1.00         2
           8       1.00      1.00      1.00         1
           9       1.00      1.00      1.00         1
          10       1.00      1.00      1.00         1
          11       1.00      1.00      1.00         1
          12       1.00      1.00      1.00         1
          13       1.00      1.00      1.00         1
          14       1.00      1.00      1.00         2
          15       1.00      1.00      1.00         1
          16       1.00      1.00      1.00         4
          17       1.00    

In [29]:
import pandas as pd

report_dict = classification_report(all_labels, all_predictions, output_dict=True, zero_division=0)

report_df = pd.DataFrame(report_dict).transpose()

class_rows = report_df.drop(index=["accuracy", "macro avg", "weighted avg"])
class_rows = class_rows[class_rows["support"] >= 3]


worst_classes = class_rows.sort_values("f1-score").head(10)
print(worst_classes)


     precision    recall  f1-score  support
122   1.000000  0.333333  0.500000      3.0
51    1.000000  0.333333  0.500000      3.0
40    1.000000  0.333333  0.500000      3.0
73    0.400000  0.666667  0.500000      3.0
166   0.750000  0.428571  0.545455     14.0
178   0.666667  0.500000  0.571429      4.0
95    0.461538  0.857143  0.600000      7.0
111   0.500000  0.800000  0.615385      5.0
4     1.000000  0.500000  0.666667      6.0
160   1.000000  0.500000  0.666667      4.0


In [30]:
model.save_pretrained("saved_model/biobert_icd_classifier_weighted")
tokenizer.save_pretrained("saved_model/biobert_icd_classifier_weighted")

('saved_model/biobert_icd_classifier_weighted/tokenizer_config.json',
 'saved_model/biobert_icd_classifier_weighted/special_tokens_map.json',
 'saved_model/biobert_icd_classifier_weighted/vocab.txt',
 'saved_model/biobert_icd_classifier_weighted/added_tokens.json',
 'saved_model/biobert_icd_classifier_weighted/tokenizer.json')

testing single sample

In [31]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_path = "saved_model/biobert_icd_classifier_weighted"

loaded_model = AutoModelForSequenceClassification.from_pretrained(model_path)
loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)

loaded_model.eval()

print("Model and tokenizer loaded successfully")

Model and tokenizer loaded successfully


In [32]:
sample = val_data.iloc[0]
print(sample)

Unnamed: 0                                                        4117
description           Sports physical with normal growth and develo...
medical_specialty                           Consult - History and Phy.
sample_name                                       Sports Physical - 1 
transcription        HISTORY: , This child is seen for a sports phy...
keywords                                                              
combined_text         Sports physical with normal growth and develo...
matched_code                                                   T502X6S
matching_score                                                0.560871
label                                                              166
Name: 4108, dtype: object


In [33]:
text = sample["combined_text"]

inputs = loaded_tokenizer(text, truncation=True, padding=True, max_length=256, return_tensors="pt")

print(inputs)

{'input_ids': tensor([[  101,  2865,  2952,  1114,  2999,  3213,  1105,  1718,   119,  1607,
           131,   117,  1142,  2027,  1110,  1562,  1111,   170,  2865,  2952,
           119,   117, 20121,  1348,  1607,   131,   117,  1131,  2274,  6092,
          1116,   117, 11872,   117,  1105, 11669,   119, 24347,  1218,   119,
          1144,  1336,  1129,   122,  1106,   123, 13624,   170,  1285,  1104,
          6831,   119,  1123, 15355, 14741,  1180,  1129,  1618,   119,  1131,
          1674,  1136,  3668,  1115,  1277,  3618,  1133,  1131,  7407,   180,
         21778, 19954,   119,  1123, 15631,  1116,  1132,  2999,   119, 25511,
          1123,  3307,   119,  5302,   170, 10552, 12948,   119,   117, 16700,
          1607,   131,   117,  1131,  1225,  1218,  1107,  1278,  1314,  1214,
           119,  4510,  1105,  4152,   117,  1185,  2645,   119,  1131, 12063,
          5663,  2109, 16938,   119,  1131,  1209,  1129,  1107,  5192,  3654,
          1105,  2017,  1107,  9576,  

In [34]:
import torch

with torch.no_grad():
    outputs = loaded_model(**inputs)

logits = outputs.logits
predicted_label = torch.argmax(logits, dim=1).item()

print(predicted_label)

93


In [35]:
label_to_code = {v: k for k, v in code_to_label.items()}

predicted_code = label_to_code[predicted_label]
actual_code = label_to_code[sample["label"]]

print("Predicted code:", predicted_code)
print("Actual code:", actual_code)

Predicted code: O361132
Actual code: T502X6S


In [3]:
codes_list = []
with open("data/icd10cm-codes-2026.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            code, description = line.split(maxsplit=1)
            codes_list.append((code, description))

codes_df = pd.DataFrame(codes_list, columns=["code", "code_description"])

print(codes_df.shape)

NameError: name 'pd' is not defined

In [38]:
predicted_description = codes_df[codes_df["code"] == predicted_code]["code_description"].values[0]
actual_description = codes_df[codes_df["code"] == actual_code]["code_description"].values[0]

print("Predicted:", predicted_code, "-", predicted_description)
print("Actual:", actual_code, "-", actual_description)

Predicted: O361132 - Maternal care for Anti-A sensitization, third trimester, fetus 2
Actual: T502X6S - Underdosing of carbonic-anhydrase inhibitors, benzothiadiazides and other diuretics, sequela


In [39]:
sample2 = val_data.sort_values("matching_score", ascending=False).iloc[0]
print(sample2)

Unnamed: 0                                                        3542
description           Flexible sigmoidoscopy.  Sigmoid and left col...
medical_specialty                                     Gastroenterology
sample_name                                              Flex Sig - 3 
transcription        PROCEDURE: , Flexible sigmoidoscopy.,PREOPERAT...
keywords             gastroenterology, olympus, gastroscope, rectal...
combined_text         Flexible sigmoidoscopy.  Sigmoid and left col...
matched_code                                                     K5731
matching_score                                                0.705609
label                                                               51
Name: 3533, dtype: object


In [40]:
text2 = sample2["combined_text"]
inputs2 = loaded_tokenizer(text2, truncation=True, padding=True, max_length=256, return_tensors="pt")

with torch.no_grad():
    outputs2 = loaded_model(**inputs2)

predicted_label2 = torch.argmax(outputs2.logits, dim=1).item()

predicted_code2 = label_to_code[predicted_label2]
actual_code2 = label_to_code[sample2["label"]]

predicted_description2 = codes_df[codes_df["code"] == predicted_code2]["code_description"].values[0]
actual_description2 = codes_df[codes_df["code"] == actual_code2]["code_description"].values[0]

print("Predicted:", predicted_code2, "-", predicted_description2)
print("Actual:", actual_code2, "-", actual_description2)

Predicted: K5731 - Diverticulosis of large intestine without perforation or abscess with bleeding
Actual: K5731 - Diverticulosis of large intestine without perforation or abscess with bleeding


In [41]:
sorted_val = val_data.sort_values("matching_score")
sample3 = sorted_val.iloc[len(sorted_val)//2]

print(sample3)

Unnamed: 0                                                        4386
description           Examination due to blood-borne pathogen expos...
medical_specialty                           Consult - History and Phy.
sample_name                                       Gen Med Consult - 3 
transcription        CHIEF COMPLAINT:, Blood-borne pathogen exposur...
keywords                                                           NaN
combined_text         Examination due to blood-borne pathogen expos...
matched_code                                                    O99111
matching_score                                                0.598012
label                                                              116
Name: 4377, dtype: object


In [42]:
text3 = sample3["combined_text"]
inputs3 = loaded_tokenizer(text3, truncation=True, padding=True, max_length=256, return_tensors="pt")

with torch.no_grad():
    outputs3 = loaded_model(**inputs3)

predicted_label3 = torch.argmax(outputs3.logits, dim=1).item()

predicted_code3 = label_to_code[predicted_label3]
actual_code3 = label_to_code[sample3["label"]]

predicted_description3 = codes_df[codes_df["code"] == predicted_code3]["code_description"].values[0]
actual_description3 = codes_df[codes_df["code"] == actual_code3]["code_description"].values[0]

print("Predicted:", predicted_code3, "-", predicted_description3)
print("Actual:", actual_code3, "-", actual_description3)

Predicted: O99111 - Other diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism complicating pregnancy, first trimester
Actual: O99111 - Other diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism complicating pregnancy, first trimester


In [43]:
import json

with open("saved_model/biobert_icd_classifier_weighted/label_to_code.json", "w") as f:
    json.dump(label_to_code, f)

print("Saved")

Saved


In [44]:
print(sample2["combined_text"])

 Flexible sigmoidoscopy.  Sigmoid and left colon diverticulosis; otherwise, normal flexible sigmoidoscopy to the proximal descending colon.PROCEDURE: , Flexible sigmoidoscopy.,PREOPERATIVE DIAGNOSIS:,  Rectal bleeding.,POSTOPERATIVE DIAGNOSIS:  ,Diverticulosis.,MEDICATIONS: , None.,DESCRIPTION OF PROCEDURE:  ,The Olympus gastroscope was introduced through the rectum and advanced carefully through the colon for a distance of 90 cm, reaching the proximal descending colon.  At this point, stool occupied the lumen, preventing further passage.  The colon distal to this was well cleaned out and easily visualized.  The mucosa was normal throughout the regions examined.  Numerous diverticula were seen.  There was no blood or old blood or active bleeding.  A retroflexed view of the anorectal junction showed no hemorrhoids.  He tolerated the procedure well and was sent to the recovery room.,FINAL DIAGNOSES:,1.  Sigmoid and left colon diverticulosis.,2.  Otherwise normal flexible sigmoidoscopy to

In [5]:
import pandas as pd

In [6]:
codes_list = []
with open("data/icd10cm-codes-2026.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            code, description = line.split(maxsplit=1)
            codes_list.append((code, description))

codes_df = pd.DataFrame(codes_list, columns=["code", "code_description"])

print(codes_df.shape)

(74719, 2)


In [7]:
predicted_description = codes_df[codes_df["code"] == "G43401"]["code_description"].values[0]
print(predicted_description)

Hemiplegic migraine, not intractable, with status migrainosus
